In [30]:
# %pip install torch
from transformers import pipeline
from nltk import sent_tokenize
import nltk
import torch
from glob import glob
import pandas as pd
import numpy as np

In [2]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/keshtech/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Load Model

In [3]:
model_name = "facebook/bart-large-mnli"
#device = 0 if torch.cuda.is_available() else "cpu"
device = 'cpu'

In [4]:
device

'cpu'

In [5]:
def load_model(device):
    classifier = pipeline("zero-shot-classification", model=model_name, device=device)
    return classifier

In [6]:
theme_classifier = load_model(device)

/home/keshtech/miniconda3/envs/series_analysis/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
theme_list = ["friendship","hope","sacrifice","battle","self development","betrayal","love","dialogue"]

In [8]:
theme_classifier(
    "I gave him a right hook then a left jab",
    theme_list,
    multi_label=True
)

{'sequence': 'I gave him a right hook then a left jab',
 'labels': ['battle',
  'self development',
  'hope',
  'sacrifice',
  'dialogue',
  'betrayal',
  'love',
  'friendship'],
 'scores': [0.9121248126029968,
  0.47499746084213257,
  0.08781782537698746,
  0.04499980807304382,
  0.020132731646299362,
  0.012040265835821629,
  0.004292320925742388,
  0.0028172561433166265]}

# Load Dataset

In [9]:
files = glob('../data/Subtitles/*.ass')

In [10]:
files[:5]

['../data/Subtitles/Naruto Season 7 - 177.ass',
 '../data/Subtitles/Naruto Season 2 - 40.ass',
 '../data/Subtitles/Naruto Season 1 - 23.ass',
 '../data/Subtitles/Naruto Season 6 - 156.ass',
 '../data/Subtitles/Naruto Season 4 - 96.ass']

In [11]:
with open(files[0],'r') as file:
    lines = file.readlines()
    lines = lines[27:]
    lines =  [ ",".join(line.split(',')[9:])  for line in lines ]

In [12]:
lines[:2]

['I want to try and gather\\Nthe unrestrained winds\n',
 'I’ll run toward the horizon,\\Nalongside the wave crests\n']

In [13]:
lines = [ line.replace('\\N',' ') for line in lines]

In [14]:
lines[:2]

['I want to try and gather the unrestrained winds\n',
 'I’ll run toward the horizon, alongside the wave crests\n']

In [15]:
" ".join(lines[:10])

'I want to try and gather the unrestrained winds\n I’ll run toward the horizon, alongside the wave crests\n I’ve made my decision,\n even if there’s a long way to go on that road.\n I’ll continue towards the future I’ve planned\n Time rushes us The heartbeat speeds up\n When I woke in the middle of a dream I kept searching for that same light\n Under the shining star-lit sky with countless constellations and shadows\n There was something beyond that… What were you gazing at?\n What were you gazing at?\n'

In [16]:
int(files[0].split('-')[-1].split('.')[0].strip())

177

In [17]:
def load_subtitles_dataset(dataset_path):
    subtitles_paths = glob(dataset_path+'/*.ass')

    scripts=[]
    episode_num=[]

    for path in subtitles_paths:

        #Read Lines
        with open(path,'r') as file:
            lines = file.readlines()
            lines = lines[27:]
            lines =  [ ",".join(line.split(',')[9:])  for line in lines ]
        
        lines = [ line.replace('\\N',' ') for line in lines]
        script = " ".join(lines)

        episode = int(path.split('-')[-1].split('.')[0].strip())

        scripts.append(script)
        episode_num.append(episode)

    df = pd.DataFrame.from_dict({"episode":episode_num, "script":scripts })
    return df

In [18]:
dataset_path = "../data/Subtitles"
df = load_subtitles_dataset(dataset_path)

In [19]:
df.head()

,episode,script
0,177,I want to try and gather the unrestrained wind...
1,40,"Press down hard on the gas\n That’s right, the..."
2,23,"C'mon!\n Running like a fugitive,\n Being chas..."
3,156,I want to try and gather the unrestrained wind...
4,96,We are Fighting Dreamers aiming high\n Fightin...


Run Model

In [20]:
script = df.iloc[0]['script']

In [21]:
script

'I want to try and gather the unrestrained winds\n I’ll run toward the horizon, alongside the wave crests\n I’ve made my decision,\n even if there’s a long way to go on that road.\n I’ll continue towards the future I’ve planned\n Time rushes us The heartbeat speeds up\n When I woke in the middle of a dream I kept searching for that same light\n Under the shining star-lit sky with countless constellations and shadows\n There was something beyond that… What were you gazing at?\n What were you gazing at?\n I’m exhausted!\n Important solo mission, my butt!\n They just didn’t have enough manpower for the castle restoration.\n That’s not the job of a Shinobi.\n Ow-ow… I won’t hold up if I don’t rest somewhere.\n Oh!\n I’ll take a bath. Here I go!\n Hey… I’m sounding like an old man.\n Wh-What?!\n Shadow Clone Jutsu!\n Thank you.\n Who the heck are you, mister?\n I’m a Delivery Ninja!\n Please, Mr. Postman!\n 23, 24, 25…\n 26…\n No mistake…\n Thanks to you Mister Naruto, all of the postal art

In [22]:
script_sentences = sent_tokenize(script)
script_sentences[:3]

['I want to try and gather the unrestrained winds\n I’ll run toward the horizon, alongside the wave crests\n I’ve made my decision,\n even if there’s a long way to go on that road.',
 'I’ll continue towards the future I’ve planned\n Time rushes us The heartbeat speeds up\n When I woke in the middle of a dream I kept searching for that same light\n Under the shining star-lit sky with countless constellations and shadows\n There was something beyond that… What were you gazing at?',
 'What were you gazing at?']

In [23]:
# Batch Sentence
sentence_batch_size=20
script_batches = []
for index in range(0,len(script_sentences),sentence_batch_size):
    sent = " ".join(script_sentences[index:index+sentence_batch_size])
    script_batches.append(sent)

In [24]:
script_batches[:2]

['I want to try and gather the unrestrained winds\n I’ll run toward the horizon, alongside the wave crests\n I’ve made my decision,\n even if there’s a long way to go on that road. I’ll continue towards the future I’ve planned\n Time rushes us The heartbeat speeds up\n When I woke in the middle of a dream I kept searching for that same light\n Under the shining star-lit sky with countless constellations and shadows\n There was something beyond that… What were you gazing at? What were you gazing at? I’m exhausted! Important solo mission, my butt! They just didn’t have enough manpower for the castle restoration. That’s not the job of a Shinobi. Ow-ow… I won’t hold up if I don’t rest somewhere. Oh! I’ll take a bath. Here I go! Hey… I’m sounding like an old man. Wh-What?! Shadow Clone Jutsu! Thank you. Who the heck are you, mister? I’m a Delivery Ninja! Please, Mr. Postman! 23, 24, 25…\n 26…\n No mistake…\n Thanks to you Mister Naruto, all of the postal articles are safe. By the way, what’

In [25]:
theme_output = theme_classifier(
    script_batches[:2],
    theme_list,
    multi_label=True
)

In [26]:
theme_output

[{'sequence': 'I want to try and gather the unrestrained winds\n I’ll run toward the horizon, alongside the wave crests\n I’ve made my decision,\n even if there’s a long way to go on that road. I’ll continue towards the future I’ve planned\n Time rushes us The heartbeat speeds up\n When I woke in the middle of a dream I kept searching for that same light\n Under the shining star-lit sky with countless constellations and shadows\n There was something beyond that… What were you gazing at? What were you gazing at? I’m exhausted! Important solo mission, my butt! They just didn’t have enough manpower for the castle restoration. That’s not the job of a Shinobi. Ow-ow… I won’t hold up if I don’t rest somewhere. Oh! I’ll take a bath. Here I go! Hey… I’m sounding like an old man. Wh-What?! Shadow Clone Jutsu! Thank you. Who the heck are you, mister? I’m a Delivery Ninja! Please, Mr. Postman! 23, 24, 25…\n 26…\n No mistake…\n Thanks to you Mister Naruto, all of the postal articles are safe. By t

In [27]:
# Wrangle Ouput
# battle: [0.51489498, 0.2156498]
themes = {}
for output in theme_output:
    for label,score in zip(output['labels'],output['scores']):
        if label not in themes:
            themes[label] = []
        themes[label].append(score)

In [29]:
themes

{'dialogue': [0.9834365248680115, 0.8871626257896423],
 'self development': [0.7999486327171326, 0.7575957179069519],
 'sacrifice': [0.7867300510406494, 0.9604353904724121],
 'battle': [0.5035982728004456, 0.23251692950725555],
 'hope': [0.49009671807289124, 0.20469118654727936],
 'friendship': [0.3165415823459625, 0.3242659866809845],
 'betrayal': [0.17679840326309204, 0.18697823584079742],
 'love': [0.15747353434562683, 0.11637633293867111]}

In [31]:
themes = {key: np.mean(np.array(value)) for key,value in themes.items()}

In [32]:
themes

{'dialogue': np.float64(0.9352995753288269),
 'self development': np.float64(0.7787721753120422),
 'sacrifice': np.float64(0.8735827207565308),
 'battle': np.float64(0.36805760115385056),
 'hope': np.float64(0.3473939523100853),
 'friendship': np.float64(0.3204037845134735),
 'betrayal': np.float64(0.18188831955194473),
 'love': np.float64(0.13692493364214897)}

In [33]:
def get_themes_inference(script):
    script_sentences = sent_tokenize(script)

    # Batch Sentence
    sentence_batch_size=20
    script_batches = []
    for index in range(0,len(script_sentences),sentence_batch_size):
        sent = " ".join(script_sentences[index:index+sentence_batch_size])
        script_batches.append(sent)
    
    # Run Model
    theme_output = theme_classifier(
        script_batches[:2],
        theme_list,
        multi_label=True
    )

    # Wrangle Output 
    themes = {}
    for output in theme_output:
        for label,score in zip(output['labels'],output['scores']):
            if label not in themes:
                themes[label] = []
            themes[label].append(score)

    themes = {key: np.mean(np.array(value)) for key,value in themes.items()}

    return themes

In [34]:
df = df.head(2)

In [35]:
df

,episode,script
0,177,I want to try and gather the unrestrained wind...
1,40,"Press down hard on the gas\n That’s right, the..."


In [36]:
output_themes = df['script'].apply(get_themes_inference)

In [37]:
theme_df = pd.DataFrame(output_themes.tolist())

In [38]:
theme_df

,dialogue,self development,sacrifice,battle,hope,friendship,betrayal,love
0,0.935300,0.778772,0.873583,0.368058,0.347394,0.320404,0.181888,0.136925
1,0.898511,0.671284,0.899896,0.893791,0.414528,0.120897,0.793941,0.180960


In [39]:
df[theme_df.columns] = theme_df
df

,episode,script,dialogue,self development,sacrifice,battle,hope,friendship,betrayal,love
0,177,I want to try and gather the unrestrained wind...,0.935300,0.778772,0.873583,0.368058,0.347394,0.320404,0.181888,0.136925
1,40,"Press down hard on the gas\n That’s right, the...",0.898511,0.671284,0.899896,0.893791,0.414528,0.120897,0.793941,0.180960
